### Generating 20k Records

In [0]:
from pyspark.sql import SparkSession
import random

spark = SparkSession.builder.appName("Flipkart20KPipeline").getOrCreate()

products = ["Laptop", "Mobile", "Tablet", "Watch", "Headphones"]
categories = ["Electronics", "Accessories"]
cities = ["Hyderabad", "Chennai", "Bangalore", "Mumbai", "Delhi", None]

data = []

for i in range(20000):
    order_id = random.randint(1000, 3000)  # duplicates
    customer_id = f"C{random.randint(1, 500)}"
    product = random.choice(products)
    category = random.choice(categories)
    city = random.choice(cities)
    date = f"2024-01-{random.randint(1,28):02d}"

    amount = random.choice([
        random.randint(1000, 60000),
        None,
        -random.randint(1000, 5000)
    ])

    quantity = random.randint(1, 5)

    data.append((order_id, customer_id, product, category, city, date, amount, quantity))

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

df = spark.createDataFrame(data, columns)

df.show(5)
df.printSchema()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|    2928|       C415|    Tablet|Electronics|  Chennai|2024-01-15|  2586|       2|
|    1952|       C248|     Watch|Electronics|Hyderabad|2024-01-07| -3435|       1|
|    1921|       C125|    Laptop|Accessories|    Delhi|2024-01-25| 33830|       3|
|    2980|       C462|Headphones|Electronics|    Delhi|2024-01-14| -2491|       4|
|    2335|       C315|    Laptop|Electronics|    Delhi|2024-01-23|  9045|       4|
+--------+-----------+----------+-----------+---------+----------+------+--------+
only showing top 5 rows
root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: string (nullable 

### Save as CSV (Raw Input)

In [0]:
df.write.mode("overwrite").option("header", True).csv("/Volumes/flipakart_20k/default/flipkart_raw_20k")

### Bronze Layer (Raw Data)

In [0]:
path_bronze = "/Volumes/flipakart_20k/default/bronze_table"
df_bronze = spark.read.option("header", True).csv("/Volumes/flipakart_20k/default/flipkart_raw_20k")
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(path_bronze)

### Silver Layer (Cleaning)

In [0]:
from pyspark.sql.functions import col, when
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
df_bronze = spark.read.format("delta").load(path_bronze)

### Convert Datatype

In [0]:
df_clean = df_bronze.withColumn("amount", col("amount").cast("int"))

### Handling Null Values

In [0]:
df_clean = df_clean.withColumn("amount",when(col("amount").isNull(), 0).otherwise(col("amount")))
df_clean = df_clean.fillna({"city": "Unknown"})

### Removing Negative Values

In [0]:
df_clean = df_clean.filter(col("amount") > 0)

In [0]:
## Validating Null Values
df_clean.filter(col("amount").isNull()).count()
## Checking Null in Cities
df_clean.filter(col("city").isNull()).count()

0

In [0]:
## Validating Negative Values
df_clean.filter(col("amount") < 0).show()
## Checking Duplicates
df_clean.groupBy("order_id").agg(count("*").alias("cnt")).filter(col("cnt") > 1).show()

+--------+-----------+-------+--------+----+----+------+--------+
|order_id|customer_id|product|category|city|date|amount|quantity|
+--------+-----------+-------+--------+----+----+------+--------+
+--------+-----------+-------+--------+----+----+------+--------+

+--------+---+
|order_id|cnt|
+--------+---+
+--------+---+



### Handling Duplicates and Updates

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,col
window=Window.partitionBy("order_id").orderBy(col("date").desc())
df_clean=df_clean.withColumn('row_num',row_number().over(window)).filter(col('row_num')==1).drop('row_num')

In [0]:
df_clean.show()

+--------+-----------+----------+-----------+---------+----------+------+--------+
|order_id|customer_id|   product|   category|     city|      date|amount|quantity|
+--------+-----------+----------+-----------+---------+----------+------+--------+
|    1000|       C305|Headphones|Electronics|Hyderabad|2024-01-26|  5371|       3|
|    1001|        C71|Headphones|Accessories|  Chennai|2024-01-27| 58380|       2|
|    1002|       C128|     Watch|Accessories|Hyderabad|2024-01-28| 42252|       2|
|    1003|       C240|    Laptop|Electronics|Hyderabad|2024-01-28| 49479|       3|
|    1004|       C308|    Mobile|Accessories|  Chennai|2024-01-09| 46468|       3|
|    1005|       C184|    Laptop|Accessories|  Chennai|2024-01-28| 26483|       1|
|    1006|       C316|    Tablet|Electronics|   Mumbai|2024-01-24| 40829|       3|
|    1007|       C262|    Tablet|Accessories|Hyderabad|2024-01-27| 16396|       5|
|    1008|       C407|    Laptop|Electronics|Bangalore|2024-01-21| 41143|       1|
|   

### Save Silver

In [0]:
path_silver = "/Volumes/flipakart_20k/default/silver_deltatable"
df_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(path_silver)

### Gold Layer (Business Insights)

In [0]:
from pyspark.sql.functions import sum, col
df_silver = spark.read.format("delta").load(path_silver)

### Sales By Product
### Sales by Category
### Sales by City
### Orders per Customer
### Top Customers
### Top Product

In [0]:
sales_product=df_silver.groupBy("product").agg(sum(col("amount")).alias("total_sales"))
sales_category=df_silver.groupBy('category').agg(sum(col("amount")).alias("total_sales"))
sales_city=df_silver.groupBy('city').agg(sum(col("amount")).alias("total_sales"))
order_customer=df_silver.groupBy("customer_id").agg(sum(col("amount")).alias("total_orders"))
spend_customers=df_silver.groupBy("customer_id").agg(sum(col("amount")).alias("total_spend"))
top_customers=spend_customers.orderBy(col("total_spend").desc()).limit(5)
top_product=sales_product.orderBy(col("total_sales").desc()).limit(5)

### Save Gold Layer

In [0]:
path_gold = "/Volumes/flipakart_20k/default/gold_deltatable"

sales_product.write.format("delta").mode("overwrite").save(path_gold + "/sales_product")
sales_category.write.format("delta").mode("overwrite").save(path_gold + "/sales_category")
sales_city.write.format("delta").mode("overwrite").save(path_gold + "/sales_city")
top_customers.write.format("delta").mode("overwrite").save(path_gold + "/top_customers")

In [0]:
df_bronze.count()

20000

In [0]:
df_clean.count()

1936

In [0]:
df_clean.describe().show()

+-------+-----------------+-----------+----------+-----------+---------+----------+------------------+------------------+
|summary|         order_id|customer_id|   product|   category|     city|      date|            amount|          quantity|
+-------+-----------------+-----------+----------+-----------+---------+----------+------------------+------------------+
|  count|             1936|       1936|      1936|       1936|     1936|      1936|              1936|              1936|
|   mean|2000.245867768595|       NULL|      NULL|       NULL|     NULL|      NULL|30362.095041322315|3.0361570247933884|
| stddev|579.1820852186411|       NULL|      NULL|       NULL|     NULL|      NULL|   16818.540009247|1.4034783309547985|
|    min|             1000|         C1|Headphones|Accessories|Bangalore|2024-01-01|              1005|                 1|
|    max|             3000|        C99|     Watch|Electronics|  Unknown|2024-01-28|             59966|                 5|
+-------+---------------